# Customer Retention Intelligence — Dockerization (Phase 13)

## Objective

Prove that the existing FastAPI + Streamlit application (Phases 1-12, unchanged) can
run reproducibly as two Docker containers:

```
Browser -> Streamlit container (host port 8501)
              | HTTP, using the Compose service name "api" -- never localhost
              v
          FastAPI container (host port 8000)
              |
              v
    existing MLflow-registered model / SHAP / recommendation code (unchanged)
```

This notebook does not implement, predict, explain, or recommend anything itself --
it drives the actual `docker compose` CLI (via `subprocess`) exactly as a developer
would from a terminal, and verifies the running containers with real HTTP calls. No
application code, model, or MLflow-registered state is modified by anything here.

**Prerequisite:** Docker must be installed and running, and Phases 1-10 must already
have produced `data/processed/customer_features.parquet`, `mlruns.db`, and
`mlartifacts/` locally (see `README.md`'s Docker section) -- this notebook does not
generate them.

At the end, the stack is torn down (`docker compose down`), leaving no containers
running and the built images intact.

In [1]:
import json
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def run(cmd: list[str], timeout: int = 120) -> subprocess.CompletedProcess:
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True, timeout=timeout)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    return result


PROJECT_ROOT

PosixPath('/Users/dhananjaymandal/customer-retention-intelligence')

## 1. Confirm Docker Is Available

In [2]:
version_result = run(["docker", "--version"])
compose_version_result = run(["docker", "compose", "version"])
info_result = run(["docker", "info", "--format", "{{.ServerVersion}}"])

docker_available = version_result.returncode == 0 and info_result.returncode == 0
print(f"\nDocker available and daemon reachable: {docker_available}")

if not docker_available:
    raise RuntimeError(
        "Docker is not available/running. Start Docker Desktop and re-run this notebook. "
        "Phase 13's actual container verification cannot proceed without it."
    )

$ docker --version
Docker version 29.7.2, build a7dcaa6
$ docker compose version
Docker Compose version v5.4.0
$ docker info --format {{.ServerVersion}}


29.7.2

Docker available and daemon reachable: True


## 2. Confirm Required Local Artifacts Exist

These are git-ignored, developer-generated outputs from Phases 4 and 10 -- not
something this notebook creates.

In [3]:
required_paths = {
    "data/processed/customer_features.parquet": PROJECT_ROOT / "data" / "processed" / "customer_features.parquet",
    "mlruns.db": PROJECT_ROOT / "mlruns.db",
    "mlartifacts/": PROJECT_ROOT / "mlartifacts",
}

missing = [name for name, path in required_paths.items() if not path.exists()]
for name, path in required_paths.items():
    print(f"{'OK ' if path.exists() else 'MISSING'}  {name}")

if missing:
    raise RuntimeError(
        f"Missing required local artifacts: {missing}. Run Phases 1-10 locally first "
        "(see README.md)."
    )

OK   data/processed/customer_features.parquet
OK   mlruns.db
OK   mlartifacts/


## 3. Validate the Compose Configuration

In [4]:
config_result = run(["docker", "compose", "config", "--quiet"])
print(f"\ndocker-compose.yml is valid: {config_result.returncode == 0}")

$ docker compose config --quiet

docker-compose.yml is valid: True


## 4. Build the Images

Uses the layer cache from any previous build, so this is fast unless
`requirements.txt` or the Dockerfiles changed.

In [5]:
build_result = run(["docker", "compose", "build"], timeout=600)
print(f"\nBuild succeeded: {build_result.returncode == 0}")

$ docker compose build


#1 [internal] load local bake definitions
#1 reading from stdin 1.14kB done
#1 DONE 0.0s

#2 [api internal] load build definition from Dockerfile.api
#2 transferring dockerfile: 2.21kB 0.0s done
#2 DONE 0.0s

#3 [dashboard internal] load build definition from Dockerfile.dashboard
#3 transferring dockerfile: 2.04kB 0.0s done
#3 DONE 0.0s

#4 [dashboard internal] load metadata for docker.io/library/python:3.14-slim
#4 DONE 0.1s

#5 [api internal] load .dockerignore
#5 transferring context: 1.25kB done
#5 DONE 0.0s

#6 [api 1/6] FROM docker.io/library/python:3.14-slim@sha256:caaf356f40667c496d405780745b9ac25771c189a51dfcc42430d531ea09f8a2
#6 resolve docker.io/library/python:3.14-slim@sha256:caaf356f40667c496d405780745b9ac25771c189a51dfcc42430d531ea09f8a2 done
#6 DONE 0.0s

#7 [api internal] load build context
#7 transferring context: 27.92kB 0.0s done
#7 DONE 0.0s

#8 [api 2/6] WORKDIR /app
#8 CACHED

#9 [api 3/6] COPY requirements.txt .
#9 CACHED

#10 [dashboard 4/6] RUN pip install --no

## 5. Start the Stack

In [6]:
up_result = run(["docker", "compose", "up", "-d"], timeout=120)
print(f"\nStack started: {up_result.returncode == 0}")

$ docker compose up -d


Network customer-retention-intelligence_default Creating 
 Network customer-retention-intelligence_default Creating 
 Network customer-retention-intelligence_default Created 
 Network customer-retention-intelligence_default Created 
 Container customer-retention-intelligence-api-1 Creating 
 Container customer-retention-intelligence-api-1 Created 
 Container customer-retention-intelligence-dashboard-1 Creating 
 Container customer-retention-intelligence-dashboard-1 Created 
 Container customer-retention-intelligence-api-1 Starting 
 Container customer-retention-intelligence-api-1 Started 
 Container customer-retention-intelligence-api-1 Waiting 
 Container customer-retention-intelligence-api-1 Healthy 
 Container customer-retention-intelligence-dashboard-1 Starting 
 Container customer-retention-intelligence-dashboard-1 Started

Stack started: True


## 6. Wait for Both Containers to Report Healthy

In [7]:
def get_service_status(service: str) -> str:
    result = run(["docker", "compose", "ps", "--format", "json", service], timeout=15)
    if not result.stdout.strip():
        return "not found"
    row = json.loads(result.stdout.strip().splitlines()[0])
    return row.get("Health") or row.get("State", "unknown")


max_wait_seconds = 90
poll_interval_seconds = 3
elapsed = 0
statuses = {}

while elapsed <= max_wait_seconds:
    statuses = {"api": get_service_status("api"), "dashboard": get_service_status("dashboard")}
    print(f"[{elapsed}s] {statuses}")
    if statuses.get("api") == "healthy" and statuses.get("dashboard") == "healthy":
        break
    time.sleep(poll_interval_seconds)
    elapsed += poll_interval_seconds

both_healthy = statuses.get("api") == "healthy" and statuses.get("dashboard") == "healthy"
print(f"\nBoth containers healthy: {both_healthy}")
if not both_healthy:
    run(["docker", "compose", "logs", "--tail", "40"])
    raise RuntimeError(f"Containers did not become healthy in time: {statuses}")

$ docker compose ps --format json api
{"Command":"\"uvicorn api.main:ap…\"","CreatedAt":"2026-09-23 15:51:42 +0530 IST","ExitCode":0,"Health":"healthy","ID":"01c3e5bc912c","Image":"customer-retention-intelligence-api","Labels":"com.docker.compose.depends_on=,com.docker.compose.project.config_files=/Users/dhananjaymandal/customer-retention-intelligence/docker-compose.yml,com.docker.compose.project.working_dir=/Users/dhananjaymandal/customer-retention-intelligence,desktop.docker.io/binds/0/Target=/app/data/processed,desktop.docker.io/binds/2/SourceKind=hostFile,desktop.docker.io/binds/3/SourceKind=hostFile,com.docker.compose.config-hash=6ee781731d542c23299aca74831cadfdeebe04b124a168ba7d3eb09bfe5fdf88,com.docker.compose.project=customer-retention-intelligence,desktop.docker.io/binds/0/Source=/Users/dhananjaymandal/customer-retention-intelligence/data/processed,desktop.docker.io/binds/1/Source=/Users/dhananjaymandal/customer-retention-intelligence/mlruns.db,desktop.docker.io/binds/1/Source

{"Command":"\"streamlit run dashb…\"","CreatedAt":"2026-09-23 15:51:43 +0530 IST","ExitCode":0,"Health":"starting","ID":"e4df71011635","Image":"customer-retention-intelligence-dashboard","Labels":"com.docker.compose.oneoff=False,com.docker.compose.project=customer-retention-intelligence,com.docker.compose.version=5.4.0,com.docker.compose.project.config_files=/Users/dhananjaymandal/customer-retention-intelligence/docker-compose.yml,com.docker.compose.service=dashboard,desktop.docker.io/ports/8501/tcp=:8501,com.docker.compose.container-number=1,com.docker.compose.project.working_dir=/Users/dhananjaymandal/customer-retention-intelligence,desktop.docker.io/binds/0/Target=/app/data/processed,desktop.docker.io/ports.scheme=v2,com.docker.compose.config-hash=6c4b6c6b85a1b5c79a157861492605cc75e4366e2d002198dbcd25de23485c8b,com.docker.compose.depends_on=api:service_healthy:false,com.docker.compose.image=sha256:381a5245237d3148eaec3c1b6578e46d97a7ed0df1e80e3dec96a2d3b0276b93,desktop.docker.io/bin

$ docker compose ps --format json api
{"Command":"\"uvicorn api.main:ap…\"","CreatedAt":"2026-09-23 15:51:42 +0530 IST","ExitCode":0,"Health":"healthy","ID":"01c3e5bc912c","Image":"customer-retention-intelligence-api","Labels":"desktop.docker.io/binds/3/SourceKind=hostFile,desktop.docker.io/ports.scheme=v2,com.docker.compose.config-hash=6ee781731d542c23299aca74831cadfdeebe04b124a168ba7d3eb09bfe5fdf88,com.docker.compose.project=customer-retention-intelligence,desktop.docker.io/binds/0/Source=/Users/dhananjaymandal/customer-retention-intelligence/data/processed,desktop.docker.io/binds/0/SourceKind=hostFile,desktop.docker.io/binds/0/Target=/app/data/processed,desktop.docker.io/binds/3/Source=/Users/dhananjaymandal/customer-retention-intelligence/mlartifacts,com.docker.compose.depends_on=,com.docker.compose.service=api,com.docker.compose.version=5.4.0,desktop.docker.io/binds/1/Source=/Users/dhananjaymandal/customer-retention-intelligence/mlruns.db,desktop.docker.io/binds/2/Target=/app/mlar

$ docker compose ps --format json api
{"Command":"\"uvicorn api.main:ap…\"","CreatedAt":"2026-09-23 15:51:42 +0530 IST","ExitCode":0,"Health":"healthy","ID":"01c3e5bc912c","Image":"customer-retention-intelligence-api","Labels":"com.docker.compose.project.config_files=/Users/dhananjaymandal/customer-retention-intelligence/docker-compose.yml,com.docker.compose.project.working_dir=/Users/dhananjaymandal/customer-retention-intelligence,desktop.docker.io/binds/0/Source=/Users/dhananjaymandal/customer-retention-intelligence/data/processed,desktop.docker.io/binds/0/SourceKind=hostFile,desktop.docker.io/binds/2/Source=/Users/dhananjaymandal/customer-retention-intelligence/mlartifacts,desktop.docker.io/binds/3/Source=/Users/dhananjaymandal/customer-retention-intelligence/mlartifacts,desktop.docker.io/ports.scheme=v2,com.docker.compose.image=sha256:572a1525951b4fd8e58d3ff159eeabc808e95d4213060b589e5d3b0b8fa0d3c0,com.docker.compose.oneoff=False,com.docker.compose.project=customer-retention-intell

{"Command":"\"streamlit run dashb…\"","CreatedAt":"2026-09-23 15:51:43 +0530 IST","ExitCode":0,"Health":"healthy","ID":"e4df71011635","Image":"customer-retention-intelligence-dashboard","Labels":"com.docker.compose.container-number=1,com.docker.compose.config-hash=6c4b6c6b85a1b5c79a157861492605cc75e4366e2d002198dbcd25de23485c8b,com.docker.compose.image=sha256:381a5245237d3148eaec3c1b6578e46d97a7ed0df1e80e3dec96a2d3b0276b93,com.docker.compose.project=customer-retention-intelligence,com.docker.compose.project.config_files=/Users/dhananjaymandal/customer-retention-intelligence/docker-compose.yml,com.docker.compose.project.working_dir=/Users/dhananjaymandal/customer-retention-intelligence,desktop.docker.io/binds/0/SourceKind=hostFile,desktop.docker.io/binds/0/Target=/app/data/processed,desktop.docker.io/ports.scheme=v2,desktop.docker.io/ports/8501/tcp=:8501,com.docker.compose.depends_on=api:service_healthy:false,com.docker.compose.oneoff=False,com.docker.compose.service=dashboard,com.docke

## 7. Verify the API From the Host

In [8]:
import requests

health_response = requests.get("http://localhost:8000/health", timeout=10)
print(f"GET /health -> {health_response.status_code}")
print(health_response.json())

model_info_response = requests.get("http://localhost:8000/model-info", timeout=10)
print(f"\nGET /model-info -> {model_info_response.status_code}")
print(model_info_response.json())

assert health_response.status_code == 200 and health_response.json()["status"] == "healthy"
assert model_info_response.status_code == 200

GET /health -> 200
{'status': 'healthy', 'model': 'customer-churn-logistic-regression', 'model_version': '1', 'detail': None}

GET /model-info -> 200
{'model_name': 'customer-churn-logistic-regression', 'model_version': '1', 'model_type': 'LogisticRegression', 'C': 0.01, 'class_weight': 'balanced'}


## 8. Verify Streamlit Is Reachable From the Host

In [9]:
streamlit_response = requests.get("http://localhost:8501/_stcore/health", timeout=10)
print(f"GET /_stcore/health -> {streamlit_response.status_code} {streamlit_response.text!r}")
assert streamlit_response.status_code == 200

GET /_stcore/health -> 200 'ok'


## 9. Verify Container-to-Container Networking

From inside the `dashboard` container: `api:8000` must work, and `localhost:8000`
must **not** (it would refer to the dashboard container itself).

In [10]:
exec_api_name = run([
    "docker", "compose", "exec", "-T", "dashboard", "python", "-c",
    "import urllib.request; print(urllib.request.urlopen('http://api:8000/health', timeout=5).read().decode())",
])
print(f"via 'api' service name succeeded: {exec_api_name.returncode == 0}")

exec_localhost = run([
    "docker", "compose", "exec", "-T", "dashboard", "python", "-c",
    "import urllib.request\ntry:\n    urllib.request.urlopen('http://localhost:8000/health', timeout=3)\n    print('UNEXPECTED SUCCESS')\nexcept Exception as e:\n    print('failed as expected:', type(e).__name__)",
])
print(f"via 'localhost' correctly failed: {'failed as expected' in exec_localhost.stdout}")

assert exec_api_name.returncode == 0
assert "failed as expected" in exec_localhost.stdout

$ docker compose exec -T dashboard python -c import urllib.request; print(urllib.request.urlopen('http://api:8000/health', timeout=5).read().decode())


{"status":"healthy","model":"customer-churn-logistic-regression","model_version":"1","detail":null}
via 'api' service name succeeded: True
$ docker compose exec -T dashboard python -c import urllib.request
try:
    urllib.request.urlopen('http://localhost:8000/health', timeout=3)
    print('UNEXPECTED SUCCESS')
except Exception as e:
    print('failed as expected:', type(e).__name__)


failed as expected: URLError
via 'localhost' correctly failed: True


## 10. Real Customer Analysis Through the Full Stack

A real row from `customer_features.parquet`, converted by the existing
`dashboard.api_client.build_customer_payload`, sent to the containerized API's
`POST /customer-analysis`.

In [11]:
from dashboard import api_client

features_df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "customer_features.parquet")
sample_row = features_df.sample(n=1, random_state=2026).iloc[0]
payload = api_client.build_customer_payload(sample_row)

import os
os.environ["API_BASE_URL"] = "http://localhost:8000"
result = api_client.customer_analysis(payload, top_k=5)

print(f"ok={result.ok}, status_code={result.status_code}")
assert result.ok

analysis = result.data
print(f"\ncustomer_id: {analysis['customer']['customer_id']}")
print(f"churn_probability: {analysis['prediction']['churn_probability']:.4f}")
print(f"risk_level: {analysis['prediction']['risk_level']}")
print(f"top_positive_drivers: {[d['feature'] for d in analysis['explanation']['top_positive_drivers']]}")
print(f"primary_action: {analysis['recommendation']['primary_action']}")
print(f"priority: {analysis['recommendation']['priority']}")
print(f"supporting_features: {analysis['recommendation']['supporting_features'][:3]}")

for key in ("prediction", "explanation", "recommendation"):
    assert key in analysis
assert analysis["explanation"]["top_positive_drivers"] or analysis["explanation"]["top_negative_drivers"]

ok=True, status_code=200

customer_id: C000874
churn_probability: 0.5587
risk_level: high
top_positive_drivers: ['active_days_90d', 'avg_pages_viewed_90d', 'txn_purchase_count_90d', 'txn_amount_sum_180d', 'txn_amount_mean_90d']
primary_action: No intervention
priority: medium
supporting_features: ['model_signal:active_days_90d', 'model_signal:avg_pages_viewed_90d', 'model_signal:txn_purchase_count_90d']


## 11. Failure Handling: Stop the API, Confirm the Dashboard Stays Up

Then restore the normal configuration.

In [12]:
run(["docker", "compose", "stop", "api"])
time.sleep(2)

dashboard_status_while_api_down = get_service_status("dashboard")
print(f"dashboard status while api is stopped: {dashboard_status_while_api_down}")

try:
    requests.get("http://localhost:8000/health", timeout=3)
    api_unreachable = False
except requests.exceptions.RequestException:
    api_unreachable = True
print(f"api correctly unreachable: {api_unreachable}")

assert dashboard_status_while_api_down in ("healthy", "running")
assert api_unreachable

$ docker compose stop api


Container customer-retention-intelligence-api-1 Stopping 
 Container customer-retention-intelligence-api-1 Stopped


$ docker compose ps --format json dashboard
{"Command":"\"streamlit run dashb…\"","CreatedAt":"2026-09-23 15:51:43 +0530 IST","ExitCode":0,"Health":"healthy","ID":"e4df71011635","Image":"customer-retention-intelligence-dashboard","Labels":"com.docker.compose.version=5.4.0,com.docker.compose.config-hash=6c4b6c6b85a1b5c79a157861492605cc75e4366e2d002198dbcd25de23485c8b,com.docker.compose.project.working_dir=/Users/dhananjaymandal/customer-retention-intelligence,desktop.docker.io/binds/0/Source=/Users/dhananjaymandal/customer-retention-intelligence/data/processed,com.docker.compose.container-number=1,com.docker.compose.service=dashboard,desktop.docker.io/binds/0/Target=/app/data/processed,com.docker.compose.image=sha256:381a5245237d3148eaec3c1b6578e46d97a7ed0df1e80e3dec96a2d3b0276b93,com.docker.compose.project=customer-retention-intelligence,com.docker.compose.project.config_files=/Users/dhananjaymandal/customer-retention-intelligence/docker-compose.yml,desktop.docker.io/binds/0/SourceKind

In [13]:
restart_result = run(["docker", "compose", "start", "api"])

elapsed = 0
while elapsed <= max_wait_seconds:
    status = get_service_status("api")
    print(f"[{elapsed}s] api status: {status}")
    if status == "healthy":
        break
    time.sleep(poll_interval_seconds)
    elapsed += poll_interval_seconds

assert get_service_status("api") == "healthy"
print("\napi restored to healthy.")

$ docker compose start api


Container customer-retention-intelligence-api-1 Starting 
 Container customer-retention-intelligence-api-1 Started
$ docker compose ps --format json api
{"Command":"\"uvicorn api.main:ap…\"","CreatedAt":"2026-09-23 15:51:42 +0530 IST","ExitCode":0,"Health":"starting","ID":"01c3e5bc912c","Image":"customer-retention-intelligence-api","Labels":"com.docker.compose.project.working_dir=/Users/dhananjaymandal/customer-retention-intelligence,desktop.docker.io/binds/0/Source=/Users/dhananjaymandal/customer-retention-intelligence/data/processed,desktop.docker.io/binds/1/SourceKind=hostFile,desktop.docker.io/binds/2/Source=/Users/dhananjaymandal/customer-retention-intelligence/mlartifacts,desktop.docker.io/binds/2/SourceKind=hostFile,desktop.docker.io/binds/3/Source=/Users/dhananjaymandal/customer-retention-intelligence/mlartifacts,desktop.docker.io/binds/3/SourceKind=hostFile,com.docker.compose.config-hash=6ee781731d542c23299aca74831cadfdeebe04b124a168ba7d3eb09bfe5fdf88,desktop.docker.io/binds/2

$ docker compose ps --format json api
{"Command":"\"uvicorn api.main:ap…\"","CreatedAt":"2026-09-23 15:51:42 +0530 IST","ExitCode":0,"Health":"starting","ID":"01c3e5bc912c","Image":"customer-retention-intelligence-api","Labels":"desktop.docker.io/binds/0/Target=/app/data/processed,desktop.docker.io/binds/3/Source=/Users/dhananjaymandal/customer-retention-intelligence/mlartifacts,desktop.docker.io/binds/3/Target=/Users/dhananjaymandal/customer-retention-intelligence/mlartifacts,desktop.docker.io/ports.scheme=v2,desktop.docker.io/ports/8000/tcp=:8000,com.docker.compose.config-hash=6ee781731d542c23299aca74831cadfdeebe04b124a168ba7d3eb09bfe5fdf88,com.docker.compose.container-number=1,desktop.docker.io/binds/1/Source=/Users/dhananjaymandal/customer-retention-intelligence/mlruns.db,desktop.docker.io/binds/1/Target=/app/mlruns.db,desktop.docker.io/binds/2/Target=/app/mlartifacts,com.docker.compose.project=customer-retention-intelligence,com.docker.compose.project.working_dir=/Users/dhananjaym

$ docker compose ps --format json api
{"Command":"\"uvicorn api.main:ap…\"","CreatedAt":"2026-09-23 15:51:42 +0530 IST","ExitCode":0,"Health":"healthy","ID":"01c3e5bc912c","Image":"customer-retention-intelligence-api","Labels":"desktop.docker.io/binds/0/SourceKind=hostFile,desktop.docker.io/binds/1/SourceKind=hostFile,desktop.docker.io/binds/1/Target=/app/mlruns.db,com.docker.compose.container-number=1,com.docker.compose.depends_on=,com.docker.compose.project.config_files=/Users/dhananjaymandal/customer-retention-intelligence/docker-compose.yml,com.docker.compose.service=api,desktop.docker.io/binds/0/Target=/app/data/processed,desktop.docker.io/binds/1/Source=/Users/dhananjaymandal/customer-retention-intelligence/mlruns.db,desktop.docker.io/binds/2/Source=/Users/dhananjaymandal/customer-retention-intelligence/mlartifacts,desktop.docker.io/binds/2/SourceKind=hostFile,com.docker.compose.image=sha256:572a1525951b4fd8e58d3ff159eeabc808e95d4213060b589e5d3b0b8fa0d3c0,desktop.docker.io/binds/

## 12. Tear Down

Stops and removes the containers/network. Built images and local `mlruns.db` /
`mlartifacts/` / `data/` are left untouched.

In [14]:
down_result = run(["docker", "compose", "down"], timeout=60)
print(f"\nStack stopped cleanly: {down_result.returncode == 0}")

$ docker compose down


Container customer-retention-intelligence-dashboard-1 Stopping 
 Container customer-retention-intelligence-dashboard-1 Stopped 
 Container customer-retention-intelligence-dashboard-1 Removing 
 Container customer-retention-intelligence-dashboard-1 Removed 
 Container customer-retention-intelligence-api-1 Stopping 
 Container customer-retention-intelligence-api-1 Stopped 
 Container customer-retention-intelligence-api-1 Removing 
 Container customer-retention-intelligence-api-1 Removed 
 Network customer-retention-intelligence_default Removing 
 Network customer-retention-intelligence_default Removed

Stack stopped cleanly: True


## Summary

- Both images (`Dockerfile.api`, `Dockerfile.dashboard`) built successfully.
- Both containers reached a healthy state via their existing health mechanisms
  (`GET /health` for the API, Streamlit's built-in `/_stcore/health`).
- The `dashboard` container reached the `api` container using the Compose service
  name, and confirmed `localhost` does **not** work for that purpose inside a
  container.
- A real customer/snapshot produced a complete prediction + SHAP explanation +
  recommendation through the fully containerized stack.
- Stopping the `api` container left the dashboard running and cleanly reporting the
  outage; restarting `api` restored normal operation.
- The stack was torn down cleanly at the end.

**No model was retrained, tuned, or modified. No SHAP or recommendation logic was
duplicated inside the dashboard container.**